In [130]:
import os, glob
import sys
import settings
import json
import concurrent.futures
from google.cloud import pubsub_v1
import google.auth
import subprocess as sp
import time
import utils
import base64
import re

In [132]:
def extract_text(payload):
    if 'parts' in payload:
        return ''.join(map(extract_text, payload['parts']))
    elif payload.get('mimeType') == 'text/plain':
        data = payload.get('body', {}).get('data', '')
        if data:
            return base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
    return ''

In [133]:
list_res = utils.service.users().messages().list(userId='me', q='in:inbox', maxResults=1).execute()
messages = list_res.get('messages', [])
eid = messages[0]['id']
msg = utils.service.users().messages().get(
    userId='me', id=eid, format='full'
).execute()

email_from = ','.join([x['value'] for x in msg['payload']['headers'] if x['name'] == 'From'])

if 'info@account.netflix.com' not in email_from:
    # return
    exit(1)

In [141]:
payload = extract_text(msg['payload'])
link = re.search(r'(?<=Get Code\r\n\[)https://.*?(?=[ \]])', payload).group()
message = f'Netflix Code: {link}'
assert len(set("\n'\"") - set(message)) == 3

In [144]:
settings

<module 'settings' from '/Users/ginoprasad/Scripts/EmailManager/settings.py'>

In [ ]:
sp.run(f"'{settings.send_text_path}' '{settings.group_chat_id}' '{message}'"

In [138]:
message

'Netflix Code: https://www.netflix.com/account/travel/verify?nftoken=Bgj8vOvcAxK5AcbAqB/ZFWy0aOAWf0KknI7z2N7Xt52O5FNZtJPWhrEHNTKCoYk707QyBaADucEDvsJcKZr/56sjI/3NFhPzEqXFcvgTnV9jpIk2fvK4u6TCFcPrYOAkMFzrSl+80I44BHuP1Rgn0SiS43vDPzeC5/ktYwI6hA6vHv1ccShSXJx4H2HviqOz7Rm8jMWus8I4x/wLh3yoTcc8RDIVdPHmS5pXBiIlm0bd3aGWJL2+t7R98CQMvw71tZiFGAYiDgoMq9Ex2DyQ4knZjpQu&messageGuid=8e31d0a5-d3ef-4ce2-bee6-b4a064eef1f8'